# 50 — Revision Summary Figures

```text
Reviewer concern addressed: Assemble already-mature, already-validated revision outputs
    into manuscript/supplement-ready summary figures. Implements frozen-plan Phase 5
    (revision/strategy/20260721_frozen_revision_plan.md, Section 3, Phase 5: Assembly).
Input files: existing PNG outputs from notebooks 10, 11, 20, 21 (paths listed in Section 2
    below). No raw data files are read.
HiMaLAYAS version: N/A -- this notebook does not import HiMaLAYAS. It only loads and
    lays out already-generated PNG images; no clustering, enrichment, or statistical
    computation is performed here.
Random seed: not applicable -- no stochastic computation, only deterministic image
    loading and figure layout.
Primary parameters: 2x2 grid layout; panel labels A-D with concise subtitles; output DPI
    for the rasterized composite.
Outputs written:
    revision/outputs/figures/50_revision_summary_figures/validation_composite_2x2.png
    revision/outputs/figures/50_revision_summary_figures/validation_composite_2x2.pdf
    revision/outputs/manifests/50_revision_summary_figures_manifest.json
Interpretation: see the readiness summary in the final cell.
```

## Scope and framing

This notebook is **figure assembly and polish, not new scientific analysis**. It does not
rerun notebooks 10, 11, 20, or 21, does not recompute any statistic, and does not modify any
source figure on disk. It reads four already-generated PNG outputs and lays them out as one
labeled 2x2 composite figure, suitable as a supplementary figure.

If a source figure is missing, undersized, or otherwise visually incompatible, this notebook
records that as a caveat in its manifest rather than silently forcing a bad composite or
rerunning the source notebook.

## 1. Locate repo root and import shared revision utilities

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    markers = ("himalayas_src", "data", ".git", "revision")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(f"Could not locate himalayas-publication repo root above {start}")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "revision" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Kernel CWD:  {Path.cwd()}")
print(f"Repo root:   {REPO_ROOT}")

Kernel CWD:  /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/notebooks
Repo root:   /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication


In [2]:
import json

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

import revision_utils as ru
from revision_utils import revision_layout

layout = revision_layout(REPO_ROOT)
for key in ("manifests_dir", "figures_dir"):
    layout[key].mkdir(parents=True, exist_ok=True)

NB_ID = "50_revision_summary_figures"
NB_FIGURES_DIR = layout["figures_dir"] / NB_ID
NB_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Figures ->", NB_FIGURES_DIR)
print("Manifests ->", layout["manifests_dir"])

Figures -> /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/figures/50_revision_summary_figures
Manifests -> /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/manifests


## 2. Define source figures and verify they exist

Four already-generated validation figures, one per notebook, each answering a distinct
Go/No-Go question (see `revision/strategy/20260721_gonogo_checkpoint_summary.md`). These are
generic statistical summary plots (retention curves, null-distribution histograms) -- not
HiMaLAYAS matrix/dendrogram renderings -- and are appropriately generic for what they show
(see `revision/notes/20260723_himalayas_native_figure_audit.md`).

In [3]:
SOURCE_FIGURES = [
    {
        "panel": "A",
        "subtitle": "Noise robustness",
        "path": layout["figures_dir"] / "10_noise_robustness_gi_pcc" / "stability_curves.png",
        "source_notebook": "10_noise_robustness_gi_pcc.ipynb",
    },
    {
        "panel": "B",
        "subtitle": "Threshold sensitivity",
        "path": layout["figures_dir"]
        / "11_depth_threshold_sensitivity"
        / "threshold_sensitivity.png",
        "source_notebook": "11_depth_threshold_sensitivity.ipynb",
    },
    {
        "panel": "C",
        "subtitle": "Annotation-label null",
        "path": layout["figures_dir"] / "20_annotation_permutation_null" / "null_vs_observed.png",
        "source_notebook": "20_annotation_permutation_null.ipynb",
    },
    {
        "panel": "D",
        "subtitle": "Same-size cluster null",
        "path": layout["figures_dir"] / "21_cluster_randomization_null" / "null_vs_observed.png",
        "source_notebook": "21_cluster_randomization_null.ipynb",
    },
]

all_sources_exist = True
for entry in SOURCE_FIGURES:
    path = entry["path"]
    exists = path.exists()
    entry["exists"] = exists
    all_sources_exist = all_sources_exist and exists
    if exists:
        # image_dimensions returns (height, width); report as (width, height) for readability.
        h, w = ru.image_dimensions(path)
        entry["source_dimensions_px"] = {"width": w, "height": h}
        entry["source_aspect_ratio"] = round(w / h, 3)
    else:
        entry["source_dimensions_px"] = None
        entry["source_aspect_ratio"] = None
    panel_id = entry["panel"]
    subtitle = entry["subtitle"]
    dims = entry.get("source_dimensions_px")
    aspect = entry.get("source_aspect_ratio")
    print(
        f"Panel {panel_id} ({subtitle}): "
        f"exists={exists} dims={dims} aspect={aspect}  <- {path.relative_to(REPO_ROOT)}"
    )

print(f"\nAll four source figures exist: {all_sources_exist}")
if not all_sources_exist:
    raise FileNotFoundError(
        "One or more source validation figures are missing -- this notebook does not "
        "rerun source notebooks to regenerate them. See the manifest for details."
    )

Panel A (Noise robustness): exists=True dims={'width': 2979, 'height': 873} aspect=3.412  <- revision/outputs/figures/10_noise_robustness_gi_pcc/stability_curves.png
Panel B (Threshold sensitivity): exists=True dims={'width': 2310, 'height': 857} aspect=2.695  <- revision/outputs/figures/11_depth_threshold_sensitivity/threshold_sensitivity.png
Panel C (Annotation-label null): exists=True dims={'width': 2479, 'height': 955} aspect=2.596  <- revision/outputs/figures/20_annotation_permutation_null/null_vs_observed.png
Panel D (Same-size cluster null): exists=True dims={'width': 2479, 'height': 955} aspect=2.596  <- revision/outputs/figures/21_cluster_randomization_null/null_vs_observed.png

All four source figures exist: True


## 3. Assemble the 2x2 composite

Each source PNG is loaded as-is and placed in its own panel via `imshow`, which preserves
the image's native pixel aspect ratio (no stretching/distortion). No source image is cropped;
`bbox_inches="tight"` on save trims only the surrounding whitespace of the assembled
composite, not any content inside a source panel. Because the four source figures have
different native aspect ratios (roughly 2.6:1 to 3.4:1, printed below), each panel keeps some
empty margin within its grid cell rather than being forced to a uniform box -- this preserves
every axis, legend, and label already baked into the source PNGs.

In [ ]:
PANEL_LABEL_FONTSIZE = 20
SUBTITLE_FONTSIZE = 13
FIG_TITLE_FONTSIZE = 15
INK_PRIMARY = "#0b0b0b"

# Source panels are wide and short (aspect ~2.6-3.4:1); figure height and spacing are
# tuned so each grid cell's own aspect ratio is close to that, minimizing empty margin
# within cells without cropping or stretching any source image.
fig, axes = plt.subplots(2, 2, figsize=(18, 7.6))
fig.subplots_adjust(left=0.02, right=0.98, top=0.87, bottom=0.02, wspace=0.05, hspace=0.15)

panel_axes = {"A": axes[0, 0], "B": axes[0, 1], "C": axes[1, 0], "D": axes[1, 1]}

for entry in SOURCE_FIGURES:
    ax = panel_axes[entry["panel"]]
    img = mpimg.imread(entry["path"])
    ax.imshow(img)
    ax.axis("off")
    # Panel letter (bold, boxed) and a concise subtitle, positioned above each panel --
    # two distinct text elements so the letter and the descriptive subtitle are each
    # unambiguous on their own, matching the convention used for Figure 40. The letter
    # gets a light boxed background so it reads as a composite-level index distinct from
    # each source PNG's own internal "A./B." sub-panel labels baked into the image below
    # it (e.g. panel B's source figure has its own internal "A. Cluster count" -- the box
    # keeps the outer A-D lettering from being confused with that inner lettering).
    ax.text(
        0.0,
        1.05,
        entry["panel"],
        transform=ax.transAxes,
        fontsize=PANEL_LABEL_FONTSIZE,
        fontweight="bold",
        color=INK_PRIMARY,
        ha="left",
        va="bottom",
        bbox=dict(boxstyle="square,pad=0.25", facecolor="#f0efe9", edgecolor="none"),
    )
    ax.text(
        0.52,
        1.05,
        entry["subtitle"],
        transform=ax.transAxes,
        fontsize=SUBTITLE_FONTSIZE,
        fontweight="normal",
        color=INK_PRIMARY,
        ha="center",
        va="bottom",
    )

fig.suptitle(
    "Supplementary Figure — Validation Summary (Yeast GI-PCC Case Study)",
    fontsize=FIG_TITLE_FONTSIZE,
    y=0.99,
    color=INK_PRIMARY,
)

png_path = NB_FIGURES_DIR / "validation_composite_2x2.png"
pdf_path = NB_FIGURES_DIR / "validation_composite_2x2.pdf"
OUTPUT_DPI = 200
fig.savefig(png_path, dpi=OUTPUT_DPI, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved {png_path}")
print(f"Saved {pdf_path}")

## 4. Record output dimensions, resizing behavior, and visual caveats

In [5]:
output_h, output_w = ru.image_dimensions(png_path)
output_dimensions_px = {"width": output_w, "height": output_h}
print(f"Composite PNG dimensions: {output_dimensions_px} at {OUTPUT_DPI} DPI")

# No source image was cropped or stretched: imshow preserves each source PNG's native
# pixel aspect ratio, and bbox_inches="tight" trims only the composite's outer whitespace.
# What DOES happen is ordinary rasterization: each source panel (already 2300-3000 px wide)
# is drawn into a shared figure at OUTPUT_DPI, so the effective on-page resolution per
# panel is lower than the source PNG's native resolution. This is standard for any
# multi-panel composite and is reported here for transparency, not because it degrades
# legibility at typical print/screen size.
cropping_or_resizing_applied = (
    "No cropping. No aspect-ratio distortion (imshow preserves native pixel aspect "
    "per panel). Ordinary rasterization at 200 DPI reduces each panel's effective "
    "resolution relative to its source PNG, since four ~2500-3000px-wide sources are "
    "laid out in one shared figure -- this is standard for a multi-panel composite."
)

visual_caveats = []
aspect_ratios = [e["source_aspect_ratio"] for e in SOURCE_FIGURES]
if max(aspect_ratios) - min(aspect_ratios) > 0.5:
    visual_caveats.append(
        f"Source panels have noticeably different native aspect ratios "
        f"({min(aspect_ratios)}-{max(aspect_ratios)}), so panels are not perfectly "
        f"uniform in size within the grid -- each keeps some empty margin in its cell "
        f"rather than being stretched to match. This is a cosmetic asymmetry, not a "
        f"data or legibility problem."
    )
else:
    visual_caveats.append("Source panel aspect ratios are close enough to look uniform.")

print("Cropping/resizing note:", cropping_or_resizing_applied)
print("\nVisual caveats:")
for c in visual_caveats:
    print(" -", c)

Composite PNG dimensions: {'width': 3462, 'height': 1514} at 200 DPI
Cropping/resizing note: No cropping. No aspect-ratio distortion (imshow preserves native pixel aspect per panel). Ordinary rasterization at 200 DPI reduces each panel's effective resolution relative to its source PNG, since four ~2500-3000px-wide sources are laid out in one shared figure -- this is standard for a multi-panel composite.

Visual caveats:
 - Source panels have noticeably different native aspect ratios (2.596-3.412), so panels are not perfectly uniform in size within the grid -- each keeps some empty margin in its cell rather than being stretched to match. This is a cosmetic asymmetry, not a data or legibility problem.


## 5. Assemble and write manifest

In [ ]:
decision = "GO" if all_sources_exist else "NEEDS_POLISH"

CLAIM_BOUNDARY = (
    "This composite summarizes validation outputs for the yeast GI-PCC case study only; "
    "it does not introduce new analysis. All four source figures come from validation "
    "notebooks already decided GO (10, 11, 20, 21); this notebook performs no clustering, "
    "enrichment, or statistical computation of its own -- it only loads and lays out "
    "already-generated PNG images."
)

manifest = {
    "notebook": "50_revision_summary_figures.ipynb",
    "status": "COMPLETE",
    "decision": decision,
    "generated_at_utc": ru.utc_timestamp(),
    "repo_root": str(layout["repo_root"]),
    "source_figures": [
        {
            "panel": e["panel"],
            "subtitle": e["subtitle"],
            "source_notebook": e["source_notebook"],
            "path": str(e["path"].relative_to(REPO_ROOT)),
            "exists": e["exists"],
            "dimensions_px": e["source_dimensions_px"],
            "aspect_ratio": e["source_aspect_ratio"],
        }
        for e in SOURCE_FIGURES
    ],
    "all_source_figures_existed": all_sources_exist,
    "output_figures": {
        "validation_composite_2x2.png": {
            "path": str(png_path.relative_to(REPO_ROOT)),
            "dimensions_px": output_dimensions_px,
            "dpi": OUTPUT_DPI,
        },
        "validation_composite_2x2.pdf": {
            "path": str(pdf_path.relative_to(REPO_ROOT)),
            "dimensions_px": None,
        },
    },
    "cropping_or_resizing_applied": cropping_or_resizing_applied,
    "visual_caveats": visual_caveats,
    "claim_boundary": CLAIM_BOUNDARY,
    "next_action": (
        "Use validation_composite_2x2.png as the supplementary validation figure. "
        "If a fully uniform panel aspect ratio is desired for final typesetting, "
        "regenerate the four source figures with matched figsize in their own "
        "notebooks (10/11/20/21) -- not attempted here, out of scope for figure "
        "assembly."
    ),
}

manifest_path = layout["manifests_dir"] / f"{NB_ID}_manifest.json"
ru.write_manifest(manifest, manifest_path)
print(f"Manifest written to: {manifest_path}")

reloaded = json.loads(manifest_path.read_text())
assert reloaded == json.loads(json.dumps(manifest, default=str)), "manifest round-trip mismatch"
print("Manifest JSON round-trip verified.")

## 6. Readiness summary

In [7]:
manifest_status = manifest["status"]
manifest_decision = manifest["decision"]
print("=" * 72)
print(f"DECISION -- {NB_ID}")
print("=" * 72)
print(f"status:   {manifest_status}")
print(f"decision: {manifest_decision}")
print(f"all source figures existed: {all_sources_exist}")
print()
print("Outputs:")
print(f"  PNG:      {png_path}")
print(f"  PDF:      {pdf_path}")
print(f"  Manifest: {manifest_path}")
print()
print("Visual caveats:")
for c in visual_caveats:
    print(f"  - {c}")
print()
print("Claim boundary:")
print(f"  {CLAIM_BOUNDARY}")

DECISION -- 50_revision_summary_figures
status:   COMPLETE
decision: GO
all source figures existed: True

Outputs:
  PNG:      /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/figures/50_revision_summary_figures/validation_composite_2x2.png
  PDF:      /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/figures/50_revision_summary_figures/validation_composite_2x2.pdf
  Manifest: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/manifests/50_revision_summary_figures_manifest.json

Visual caveats:
  - Source panels have noticeably different native aspect ratios (2.596-3.412), so panels are not perfectly uniform in size within the grid -- each keeps some empty margin in its cell rather than being stretched to match. This is a cosmetic asymmetry, not a data or legibilit